In [ ]:
# Structure comparison: pLDDT vs. TM-score

# Compares the structures generated in a pipeline run (ProteinMPNN or ESM Inverse Folding, followed by ColabFold) with the reference toxins.

# The notebook reads the filtered TM-score CSV file of the selected run (`CURRENT_RUN`) and calculates the mean pLDDT of each filtered ColabFold model from the B-factor column of its CA atoms. 
# The mean pLDDT is then plotted against the TM-score (`tm1`):
# - scatter plot for the reference toxin of the run (4K09 for runs 6-10, 1KBA for runs 11-15),
# - scatter plot of all data,
# - KDE density plot (the reference toxin of the run for runs 6-15, all data otherwise).

# The structural similarity (StS) threshold is drawn at a TM-score of 0.6. All plots are saved as PDF files in `PLOTS_DIR`.

In [ ]:
# ---------------------------------------------------------------------------
# Note: This notebook was used for both the ProteinMPNN and the ESM Inverse Folding
#       pipeline runs. Only the paths and the model label (MODEL_LABEL) in the
#       Configuration section below need to be changed to point at the respective
#       pipeline's output directory and to label the plots correctly.
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# Configuration — edit these values to match your setup.
# ---------------------------------------------------------------------------

# Run number to analyze (drives the input/output filenames below).
CURRENT_RUN = 3

# Directory containing the AlphaFold/ColabFold TM-score CSV files.
BASE_DIR = "/path_to_your_base_directory/Pipeline_PMPNN_filtered"

# Directory containing the RFdiffusion (no-AlphaFold) TM-score CSV files.
NO_ALPHAFOLD_DIR = "/path_to_your_base_directory/comparison_results"

# Directory holding the filtered PDB files for the selected run.
FILTERED_PDB_DIR = f"/path_to_your_base_directory/PMPNN_filtered/run{CURRENT_RUN}_filtered"

# Directory where the generated plots (PDFs) are written.
PLOTS_DIR = "/path_to_your_base_directory/ProteinMPNN_filtered"

# Model label used in the plot legends (e.g. "PMPNN + CF" for ProteinMPNN or
# "ESM + CF" for ESM Inverse Folding; CF = ColabFold).
MODEL_LABEL = "PMPNN + CF"

# Input TM-score CSV filenames.
TM_NO_ALPHAFOLD_FILENAME = f"TM-score_{CURRENT_RUN}.csv"
TM_ALPHAFOLD_FILENAME = f"TM_score_{CURRENT_RUN}_filtered.csv"
# ---------------------------------------------------------------------------


In [ ]:
import pandas as pd
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
def get_mean_plddt(pdb_path):
    """Extract mean pLDDT score (stored in B-factor column) for CA atoms from a PDB file."""
    plddt_scores = []
    if not os.path.exists(pdb_path):
        return None
    with open(pdb_path, 'r') as f:
        for line in f:
            if line.startswith("ATOM"):
                if line[12:16].strip() == "CA":
                    plddt_scores.append(float(line[60:66].strip()))
    return sum(plddt_scores) / len(plddt_scores) if plddt_scores else None

In [ ]:
csvpath_no_alphafold = os.path.join(NO_ALPHAFOLD_DIR, TM_NO_ALPHAFOLD_FILENAME)
csvpath_alphafold = os.path.join(BASE_DIR, TM_ALPHAFOLD_FILENAME)

In [ ]:
df_without_alphafold = pd.read_csv(csvpath_no_alphafold)
df_alphafold = pd.read_csv(csvpath_alphafold, low_memory=False)

df_without_alphafold['processing_status'] = "RFdiffusion"
df_alphafold['processing_status'] = MODEL_LABEL

In [ ]:
filtered_pdb_dir = Path(FILTERED_PDB_DIR)
print("Scanning folder structure for PDB files...")
pdb_path_map = {file.name: file for file in filtered_pdb_dir.rglob("*.pdb")}
print(f"{len(pdb_path_map)} PDB files found.")

def get_plddt_fast(fname):
    pdb_path = pdb_path_map.get(fname)
    return get_mean_plddt(pdb_path) if pdb_path else None

df_alphafold['plddt'] = df_alphafold['generated_file'].apply(get_plddt_fast)

In [ ]:
print("First rows of the DataFrame:")
print(df_alphafold.head())
print("\nCount per processing status:")
print(df_alphafold['processing_status'].value_counts())

In [ ]:
# Global plot styling
plt.rcParams['axes.titlesize'] = 24
plt.rcParams['axes.labelsize'] = 22
plt.rcParams['xtick.labelsize'] = 20
plt.rcParams['ytick.labelsize'] = 20
plt.rcParams['legend.fontsize'] = 18
plt.rcParams['legend.title_fontsize'] = 18

In [ ]:
# Filter for reference protein: runs 6–10 use 4K09, runs 11–15 use 1KBA
if 6 <= CURRENT_RUN <= 10:
    df_without_alphafold_4k09 = df_without_alphafold[df_without_alphafold['reference_file'] == 'pdb4k09.ent']
    df_alphafold_4k09 = df_alphafold[df_alphafold['reference_file'] == 'pdb4k09.ent']

In [ ]:
# Scatter plot: pLDDT vs TM-score (runs 6–10, 4K09 reference)
if 6 <= CURRENT_RUN <= 10:
    plt.figure(figsize=(10, 7))
    sns.scatterplot(data=df_alphafold_4k09, x='plddt', y='tm1', color='tab:blue', label=f'{MODEL_LABEL} 4K09.pdb', alpha=0.8, s=40)
    plt.title('Protein structural similarity (StS) comparison')
    plt.xlabel('mean pLDDT')
    plt.ylabel('TM-score')
    plt.ylim(0.0, 1.0)
    plt.xlim(69, 100)
    plt.axhline(y=0.6, color="green", linestyle="--", linewidth=1.5, label="StS threshold")
    plt.legend(title='Processing status', loc='lower right')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, f"scatter_plddt_vs_tm_run{CURRENT_RUN}_4k09.pdf"), format="pdf")

In [ ]:
df_without_alphafold_1kba = df_without_alphafold[df_without_alphafold['reference_file'] == 'pdb1kba.ent']
df_alphafold_1kba = df_alphafold[df_alphafold['reference_file'] == 'pdb1kba.ent']

In [ ]:
# Scatter plot: pLDDT vs TM-score (runs 11–15, 1KBA reference)
if CURRENT_RUN > 10:
    plt.figure(figsize=(10, 7))
    sns.scatterplot(data=df_alphafold_1kba, x='plddt', y='tm1', color='tab:blue', label=f'{MODEL_LABEL} 1KBA.pdb', alpha=0.8, s=40)
    plt.title('Protein structural similarity (StS) comparison')
    plt.xlabel('mean pLDDT')
    plt.ylabel('TM-score')
    plt.ylim(0.0, 1.0)
    plt.xlim(69, 100)
    plt.axhline(y=0.6, color="green", linestyle="--", linewidth=1.5, label="StS threshold")
    plt.legend(title='Processing status', loc='lower right')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, f"scatter_plddt_vs_tm_run{CURRENT_RUN}_1kba.pdf"), format="pdf")

In [ ]:
# Scatter plot: pLDDT vs TM-score (all reference proteins)
plt.figure(figsize=(10, 7))
sns.scatterplot(data=df_alphafold, x='plddt', y='tm1', color='tab:blue', label=MODEL_LABEL, alpha=0.8, s=40)
plt.title('Protein structural similarity (StS) comparison')
plt.xlabel('mean pLDDT')
plt.ylabel('TM-score')
plt.ylim(0.0, 1.0)
plt.xlim(69, 100)
plt.axhline(y=0.6, color="green", linestyle="--", linewidth=1.5, label="StS threshold")
plt.legend(title='Processing status', loc='lower right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, f"scatter_plddt_vs_tm_run{CURRENT_RUN}.pdf"), format="pdf")

In [ ]:
# KDE plot: pLDDT vs TM-score density
if 6 <= CURRENT_RUN <= 10:
    data_to_plot, ref_name = df_alphafold_4k09, "4k09"
elif 11 <= CURRENT_RUN <= 15:
    data_to_plot, ref_name = df_alphafold_1kba, "1kba"
else:
    data_to_plot, ref_name = df_alphafold, "all"

plt.figure(figsize=(10, 7))
sns.kdeplot(data=data_to_plot, x='plddt', y='tm1', fill=True, cmap="managua",
            thresh=0.02, levels=20, clip=((69, 100), (0.0, 1.0)), cbar=True)
plt.title('KDE plot of mean pLDDT vs. TM-score')
plt.xlabel('mean pLDDT')
plt.ylabel('TM-score')
plt.ylim(0.0, 1.0)
plt.xlim(69, 100)
plt.axhline(y=0.6, color="green", linestyle="--", linewidth=1.5, label="StS threshold")
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, f"kde_plddt_vs_tm_run{CURRENT_RUN}_{ref_name}.pdf"), format="pdf")
plt.show()